# 003 · Executors with concurrent.futures

> **Standard Library › Concurrency** · Python 3.14

`concurrent.futures` is the high-level way to run work in the background: hand a function to an **executor** (a pool of threads, processes or interpreters) and get back a **`Future`** — Python's equivalent of `Task<T>`. It handles return values, exceptions and shutdown for you, so in day-to-day code you rarely touch raw `Thread` objects.

**In this notebook you will learn:**

- `ThreadPoolExecutor` vs the .NET thread pool, `Task.Run` and `Parallel.ForEach`
- `submit()` → `Future`: `result()`, `exception()`, `done()`, `add_done_callback()`
- `map()`, `as_completed()` (≈ a `Task.WhenAny` loop) and `wait(FIRST_COMPLETED)`
- how exceptions travel from workers to `.result()`
- shutting down with the `with` statement, and cancelling
- `ProcessPoolExecutor` for CPU-bound work (and why notebook functions must live in an importable module)
- `InterpreterPoolExecutor`, new in Python 3.14

## .NET mental model

| .NET | Python `concurrent.futures` |
|---|---|
| `ThreadPool` / `Task.Run(() => F(x))` | `ThreadPoolExecutor().submit(f, x)` |
| `Task<T>` | `Future` |
| `task.Result` / `await task` | `future.result()` (blocks; re-raises exceptions) |
| `task.IsCompleted` | `future.done()` |
| `task.Exception` | `future.exception()` |
| `task.ContinueWith(...)` | `future.add_done_callback(fn)` |
| `Task.WhenAll(tasks)` | `wait(fs)` / `[f.result() for f in fs]` / `executor.map` |
| `Task.WhenAny` in a loop | `as_completed(fs)` |
| `Parallel.ForEach` / PLINQ | `executor.map(fn, items)` |
| `CancellationToken` | `future.cancel()` (only before it starts) |
| *(no equivalent — .NET threads already run in parallel)* | `ProcessPoolExecutor`, `InterpreterPoolExecutor` |

The big difference: .NET `Task`s are usually awaited asynchronously. A `concurrent.futures.Future` is **blocking** — `result()` blocks the calling thread. (Async code uses `asyncio` futures instead; the two can be bridged, see *Standard Library › Concurrency › 005 · asyncio in Depth*.)

## `submit()` and `Future`

```csharp
// C#
Task<int> task = Task.Run(() => SlowSquare(4));
int value = task.Result;     // or: await task
```

`executor.submit(fn, *args, **kwargs)` schedules `fn(*args, **kwargs)` on a pool thread and immediately returns a `Future`.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def slow_square(n):
    time.sleep(0.1)          # simulated I/O
    return n * n

with ThreadPoolExecutor(max_workers=4) as executor:
    future = executor.submit(slow_square, 4)
    print("done right away?", future.done())   # False — still running
    print("result:", future.result())          # blocks until ready → 16
    print("done now?", future.done())          # True

The `with` block calls `executor.shutdown(wait=True)` on exit: it waits for all submitted work to finish and releases the threads — like disposing a custom pool. Always use it (or call `shutdown()` yourself).

**Tip:** `result(timeout=...)` raises `TimeoutError` if the result isn't ready in time — the running task is **not** cancelled, you just stop waiting.

In [ ]:
with ThreadPoolExecutor() as executor:
    f = executor.submit(slow_square, 3)
    try:
        f.result(timeout=0.01)
    except TimeoutError as e:
        print(f"{type(e).__name__}: gave up waiting, task still running? {f.running()}")
    print("eventually:", f.result())

## Running many tasks: list of futures

```csharp
// C#
var tasks = ids.Select(id => Task.Run(() => SlowSquare(id))).ToArray();
int[] results = await Task.WhenAll(tasks);
```

Submit everything first, then collect results. Collecting in submission order gives deterministic output, regardless of which task finished first.

In [ ]:
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(slow_square, n) for n in range(5)]
    results = [f.result() for f in futures]        # ≈ Task.WhenAll
elapsed = time.perf_counter() - start

print(results)
print(f"5 x 0.1s tasks took ~{elapsed:.1f}s")      # ~0.1s, not 0.5s

## `map()` — the parallel `Select`

`executor.map(fn, iterable)` is like the built-in `map`, but calls run concurrently. Results come back **in input order** (like `Parallel.ForEach` writing into an array by index, or PLINQ with `AsOrdered()`). Several iterables are zipped, like the built-in `map`.

In [ ]:
with ThreadPoolExecutor(max_workers=4) as executor:
    print(list(executor.map(slow_square, [1, 2, 3, 4])))
    print(list(executor.map(pow, [2, 3, 4], [5, 2, 3])))   # pow(2,5), pow(3,2), pow(4,3)

⚠️ **Gotcha:** `map()` returns a lazy iterator. If a call raised, the exception is re-raised **when you reach that item** while iterating — and the remaining results are lost from that loop. If you need per-item error handling, use `submit()` and handle each future.

In [ ]:
def parse_price(text):
    return float(text)

with ThreadPoolExecutor() as executor:
    results = executor.map(parse_price, ["9.99", "oops", "3.50"])
    try:
        for r in results:
            print("parsed", r)
    except ValueError as e:
        print(f"{type(e).__name__}: {e}")

## Exceptions surface on `.result()`

An exception raised in a worker is stored in the future and **re-raised** when you call `result()` — the same as `task.Result` throwing (though without .NET's `AggregateException` wrapper). `exception()` returns it without raising (or `None`).

In [ ]:
def risky(n):
    if n == 2:
        raise ValueError(f"bad input {n}")
    return n * 10

with ThreadPoolExecutor() as executor:
    futures = {n: executor.submit(risky, n) for n in [1, 2, 3]}

for n, f in futures.items():
    if (err := f.exception()) is not None:
        print(n, "failed ->", f"{type(err).__name__}: {err}")
    else:
        print(n, "ok ->", f.result())

## `as_completed()` — process results as they finish

```csharp
// C#
var pending = tasks.ToList();
while (pending.Count > 0)
{
    var finished = await Task.WhenAny(pending);
    pending.Remove(finished);
    Console.WriteLine(await finished);
}
```

`as_completed(futures)` yields each future **as soon as it completes** — exactly the `Task.WhenAny` loop above in one line. Here the delays are far enough apart that the completion order is predictable:

In [ ]:
from concurrent.futures import as_completed

def fetch(name, delay):
    time.sleep(delay)
    return name

jobs = {"slow": 0.25, "fast": 0.05, "medium": 0.15}
with ThreadPoolExecutor() as executor:
    futures = [executor.submit(fetch, name, d) for name, d in jobs.items()]
    for f in as_completed(futures):
        print("finished:", f.result())     # fast, medium, slow

**Tip:** a common pattern is a dict mapping future → input, so you know which input each completed future belongs to: `future_to_url = {executor.submit(download, u): u for u in urls}`.

## `wait()` — first completed, first exception, or all

`wait(futures, timeout=None, return_when=...)` returns a named tuple `(done, not_done)` of **sets**:

- `ALL_COMPLETED` (default) — like `Task.WhenAll`
- `FIRST_COMPLETED` — like `Task.WhenAny`
- `FIRST_EXCEPTION` — return as soon as any future raises

In [ ]:
from concurrent.futures import wait, FIRST_COMPLETED

with ThreadPoolExecutor() as executor:
    futures = [executor.submit(fetch, name, d) for name, d in jobs.items()]
    done, not_done = wait(futures, return_when=FIRST_COMPLETED)
    print("first done:", [f.result() for f in done])
    print("still pending:", len(not_done))

## Callbacks with `add_done_callback`

`add_done_callback(fn)` calls `fn(future)` when the future finishes — similar to `ContinueWith`. The callback runs on the worker thread that completed the future (or immediately on your thread, if it is already done), so keep it short and thread-safe.

In [ ]:
log = []

def report(fut):
    log.append(f"callback got {fut.result()}")

with ThreadPoolExecutor() as executor:
    for n in [3, 1, 2]:
        executor.submit(slow_square, n).add_done_callback(report)

print(sorted(log))      # sorted: completion order is not guaranteed

## Cancelling and shutting down

`future.cancel()` only succeeds if the task **hasn't started yet** — Python cannot interrupt a running thread (there is no `Thread.Abort`, and no cooperative `CancellationToken` unless you pass your own `Event`). `shutdown(cancel_futures=True)` cancels everything still waiting in the queue.

In [ ]:
with ThreadPoolExecutor(max_workers=1) as executor:   # one worker → others queue up
    running = executor.submit(slow_square, 1)
    queued = executor.submit(slow_square, 2)
    time.sleep(0.02)                                   # let the first one start
    print("cancel running:", running.cancel())         # False — already started
    print("cancel queued: ", queued.cancel())          # True — never ran
    print("queued.cancelled():", queued.cancelled())

⚠️ **Gotcha:** calling `result()` on a cancelled future raises `concurrent.futures.CancelledError`.

**Tip:** `ThreadPoolExecutor()` without `max_workers` picks a default based on the CPU count (`min(32, cpus + 4)`), which suits I/O-bound work. For lots of network calls you can go higher; for CPU-bound work threads won't help anyway (the GIL — see *Standard Library › Concurrency › 001 · Threads*).

## `ProcessPoolExecutor` — CPU-bound work on all cores

Because of the GIL, pure-Python CPU-bound code doesn't get faster with threads. `ProcessPoolExecutor` has the **same API** but runs tasks in separate worker **processes**, each with its own interpreter and GIL. The price:

- the function and its arguments/results are **pickled** and sent between processes
- process start-up costs tens of milliseconds, so tasks should be reasonably large
- the function must be **importable by name** in the worker process

On macOS and Windows, workers are started with the **spawn** method: a brand-new Python process imports your module to find the function. Functions defined in a notebook live in the kernel's `__main__`, which a worker cannot import. In a notebook it fails like this (the workers crash while unpickling the task):

```text
AttributeError: module '__main__' has no attribute 'cpu_heavy'   (printed by each worker)
BrokenProcessPool: A process in the process pool was terminated abruptly ...
```

The fix is to put worker functions in a real module. In a project that's just a `.py` file next to your code; here we write one into a temporary directory and add it to `sys.path`:

In [ ]:
import sys
import tempfile
import textwrap
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
Path(tmp.name, "cpu_tasks.py").write_text(textwrap.dedent("""
    def cpu_heavy(n):
        total = 0
        for i in range(n):
            total += i * i
        return total

    def sum_of_squares(start, stop):
        return sum(i * i for i in range(start, stop))
"""))
sys.path.insert(0, tmp.name)

import cpu_tasks
print(cpu_tasks.cpu_heavy(10))

In [ ]:
from concurrent.futures import ProcessPoolExecutor

start = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(cpu_tasks.cpu_heavy, [2_000_000] * 4))
elapsed = time.perf_counter() - start

print(results[0], len(results))
print(f"4 CPU-heavy tasks in separate processes: {elapsed:.2f}s")

On a multi-core machine the four tasks run in parallel (the total includes start-up overhead of the worker processes). Everything else — `submit`, `as_completed`, exceptions on `result()` — works exactly as with threads:

In [ ]:
with ProcessPoolExecutor(max_workers=2) as executor:
    f = executor.submit(cpu_tasks.sum_of_squares, 0, 4)
    print(f.result())               # 0 + 1 + 4 + 9 = 14

    bad = executor.submit(cpu_tasks.sum_of_squares, 0, "x")
    try:
        bad.result()
    except TypeError as e:
        print(f"{type(e).__name__}: {e}")   # the worker's exception, re-raised here

⚠️ **Gotcha:** lambdas, nested functions and objects such as open files, locks or sockets **cannot be pickled**, so they can't be sent to a process pool. Pass plain data (numbers, strings, lists, dicts, dataclasses) and module-level functions.

⚠️ **Gotcha:** in a **script**, guard the code that creates the pool with `if __name__ == "__main__":` — a spawned worker imports your main module, and without the guard it would try to create its own pool *(see Standard Library › Concurrency › 004 · Processes with multiprocessing)*.

## `InterpreterPoolExecutor` (Python 3.14)

Python 3.14 adds `InterpreterPoolExecutor`: each worker is a **sub-interpreter** in the same process, running on its own thread with **its own GIL**. So CPU-bound work runs in parallel like with processes, but with cheaper start-up and no extra OS processes. Each interpreter has its own modules and globals — nothing is shared implicitly, and arguments/results are copied between interpreters.

It is the newest of the three and the surrounding ecosystem (C extensions supporting sub-interpreters) is still catching up, so treat it as an option to evaluate rather than a default.

In [ ]:
import concurrent.futures

print("available:", hasattr(concurrent.futures, "InterpreterPoolExecutor"))

from concurrent.futures import InterpreterPoolExecutor

with InterpreterPoolExecutor(max_workers=2) as executor:
    print(list(executor.map(cpu_tasks.sum_of_squares, [0, 10], [10, 20])))

Clean up the temporary module directory:

In [ ]:
sys.path.remove(tmp.name)
del sys.modules["cpu_tasks"]
tmp.cleanup()
print("cleaned up")

## Choosing an executor

| Workload | Executor | .NET analogue |
|---|---|---|
| I/O-bound (HTTP, DB, files) | `ThreadPoolExecutor` | `Task.Run` / async I/O |
| CPU-bound pure Python | `ProcessPoolExecutor` | `Parallel.For` (threads suffice in .NET) |
| CPU-bound, want lighter isolation (3.14+) | `InterpreterPoolExecutor` | — |
| Thousands of concurrent network calls | `asyncio` instead | `async`/`await` |

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Run on pool thread | `Task.Run(() => F(x))` | `executor.submit(f, x)` |
| Get result | `await t` / `t.Result` | `future.result(timeout=...)` |
| Is it finished? | `t.IsCompleted` | `future.done()` |
| Get exception | `t.Exception` | `future.exception()` |
| Continuation | `t.ContinueWith(...)` | `future.add_done_callback(fn)` |
| Parallel map | `Parallel.ForEach` / PLINQ | `executor.map(fn, items)` |
| Wait all | `Task.WhenAll` | `wait(fs)` |
| Wait any | `Task.WhenAny` | `wait(fs, return_when=FIRST_COMPLETED)` |
| Process as finished | `WhenAny` loop | `for f in as_completed(fs):` |
| Cancel | `CancellationToken` | `future.cancel()` (not yet started only) |
| Dispose pool | — | `with Executor() as ex:` / `ex.shutdown()` |
| CPU parallelism | threads | `ProcessPoolExecutor` / `InterpreterPoolExecutor` |

## Exercises

1. Use a `ThreadPoolExecutor` and `map()` to "check" the five hosts `["h1", "h2", "h3", "h4", "h5"]` with a function that sleeps 0.05s and returns `f"{host}: up"`. Print the results and confirm it took well under 0.25s.
2. Submit `risky(n)` (raise `ValueError` for odd `n`, else return `n * 10`) for `n` in `range(6)`. Using `as_completed`, collect successful results and error messages into two lists, then print both **sorted**.
3. Submit three `fetch(name, delay)` tasks with delays 0.2, 0.05 and 0.1, and use `wait(..., return_when=FIRST_COMPLETED)` to print only the name of the winner.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import time
from concurrent.futures import ThreadPoolExecutor, as_completed, wait, FIRST_COMPLETED

# 1 — map keeps input order
def check(host):
    time.sleep(0.05)
    return f"{host}: up"

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as ex:
    print(list(ex.map(check, ["h1", "h2", "h3", "h4", "h5"])))
print("fast enough:", time.perf_counter() - start < 0.25)

# 2 — per-future error handling with as_completed
def risky(n):
    if n % 2:
        raise ValueError(f"odd {n}")
    return n * 10

ok, errors = [], []
with ThreadPoolExecutor() as ex:
    for f in as_completed([ex.submit(risky, n) for n in range(6)]):
        try:
            ok.append(f.result())
        except ValueError as e:
            errors.append(str(e))
print(sorted(ok), sorted(errors))

# 3 — first completed
def fetch(name, delay):
    time.sleep(delay)
    return name

with ThreadPoolExecutor() as ex:
    fs = [ex.submit(fetch, n, d) for n, d in [("a", 0.2), ("b", 0.05), ("c", 0.1)]]
    done, _ = wait(fs, return_when=FIRST_COMPLETED)
    print("winner:", next(iter(done)).result())
```

</details>